# For Google Colab use 2026.01 runtime

In [1]:
!pip install syncode

# Question 2

In [11]:
from syncode import Syncode
import re
import operator as op

In [3]:
def get_syncode_llm():
  grammar =  r"""
  start: operand OP operand
  operand: NUMBER | NEGNUM

  NUMBER: /[0-9]+([.][0-9]*)?([eE][-+]?[0-9]+)?/
  NEGNUM: /\(-[0-9]+([.][0-9]*)?([eE][-+]?[0-9]+)?\)/
  OP: "+" | "-" | "*" | "/"
  """
  model_name = "microsoft/phi-2"

  return Syncode(model = model_name, grammar=grammar, parse_output_only=True, max_new_tokens=20)

syn_llm = get_syncode_llm()


[2026-10-28 05:06:02,806-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:

NUM = r'[0-9]+(?:[.][0-9]*)?(?:[eE][-+]?[0-9]+)?'
OPERAND = rf'(?:{NUM}|\(-{NUM}\))'
EXPR_RE = re.compile(rf'\s*({OPERAND})\s*([-+*/])\s*({OPERAND})\s*')

OPS = {'+': op.add, '-': op.sub, '*': op.mul, '/': op.truediv}

def parse_operand(tok):
    return float(tok.strip('()'))

def evaluate(expression):
    m = EXPR_RE.search(expression)
    left, operator, right = m.groups()
    operands = [parse_operand(left), parse_operand(right)]
    try:
        result = OPS[operator](*operands)
    except ZeroDivisionError:
        result = float('nan')
    return operands, operator, result



def run_syncode(query):
  system_prompt = """
  The user will ask simple math questions like the following question and answer them to the point nothing extra.
  1. What is 10+15? you are expected to say Answer: 10 + 15.
  2. What is 10 minus 5? Answer: 10 - 5
  3. What is 10 added with 9? Answer: 10 + 9
  4. What is -10 added to 909? Answer: (-10) + 909
  5. What is -11 time negative 100? Answer: (-11) * (-100)
  6. What is -11 subtracted from -100? Answer: (-100) - (-11)
  7. What is the sum of 18 and 19? Answer: 18 + 19

  You saw the examples with answers. Now answer the user question \n
  """

  input =  system_prompt + query +"  Answer:"
  expression = syn_llm.infer(input)[0]
  print(expression)
  return evaluate(expression)

In [14]:
questions = ["what is -26 multiplied with 15 ?" ,
             'What is -20 divided by 0?',
             "What is 200 devided by one thousand?",
             "What is minus 20 divided by -0?",
             "What is two hundred divided by negative one thousand?" ]

for question in questions:
  print(question + " Answer: ", run_syncode(question))

(-26)*15
what is -26 multiplied with 15 ? Answer:  ([-26.0, 15.0], '*', -390.0)
(-20)/0
What is -20 divided by 0? Answer:  ([-20.0, 0.0], '/', nan)
200/1000
What is 200 devided by one thousand? Answer:  ([200.0, 1000.0], '/', 0.2)
(-20)/(-0)
What is minus 20 divided by -0? Answer:  ([-20.0, -0.0], '/', nan)
200/1000
What is two hundred divided by negative one thousand? Answer:  ([200.0, 1000.0], '/', 0.2)
